# Lower Bounds and No-Go Theorems

## A tour of the *constraints*. Some are mathematical (BBBV's $\sqrt N$ Grover lower bound, the Holevo bound). Some are physical (no cloning, no signalling, no fast-forwarding). Each rules out something one might hope a quantum computer could do — and together they explain why quantum advantages take the *shape* they do.

# 1. No-cloning (recap from chapter 1)

## **Theorem.** There is no unitary $U$ on $\mathcal H \otimes \mathcal H$ such that $U(|\psi\rangle \otimes |0\rangle) = |\psi\rangle \otimes |\psi\rangle$ for *every* state $|\psi\rangle$.

## **Proof (one line):** if such a $U$ existed, taking inner products of two distinct copies forces $\langle\psi|\phi\rangle = \langle\psi|\phi\rangle^2$, i.e. $\langle\psi|\phi\rangle \in \{0, 1\}$. ∎

## Consequences:

## - Quantum **error correction** cannot just duplicate the state — needs entangled encodings (chapter 9).
## - **BB84** and other QKD schemes are secure: an eavesdropper cannot intercept-and-resend without perturbing the state.
## - **Unforgeable quantum money** (Wiesner) is possible.
## - Combined with the no-signalling principle, this rules out superluminal communication via entanglement.

## **Approximate / probabilistic clones** are allowed. Optimal universal symmetric cloners (Bužek–Hillery, 1996) achieve fidelity $5/6$ for $1 \to 2$ qubit cloning — better than measurement-and-prepare, but never $1$.

# 2. No-broadcasting and no-deletion

## **No-broadcasting (Barnum–Caves–Fuchs–Jozsa–Schumacher, 1996).** Cloning's sibling for *mixed* states: there is no quantum operation that takes an unknown $\rho$ to a joint state $\sigma_{AB}$ with marginals $\sigma_A = \sigma_B = \rho$, except when the $\rho$ are drawn from a commuting family. So even *broadcasting* (the weakest sense of copying) is impossible for general mixed states.

## **No-deletion (Pati–Braunstein, 2000).** Given *two* copies of an unknown state $|\psi\rangle$, there is no unitary that sends one of them back to a fixed blank $|0\rangle$ while keeping the other intact:

$$ \Large  U(|\psi\rangle \otimes |\psi\rangle \otimes |A\rangle) \neq |\psi\rangle \otimes |0\rangle \otimes |A_\psi\rangle \quad \text{for all } \psi. $$

## Together: quantum information is neither freely copyable nor freely deletable. It behaves more like a conserved fluid than ordinary data.

# 3. No-signalling

## Alice and Bob share an entangled state. Whatever local operations Alice performs (unitaries, measurements, discarding qubits), Bob's reduced density matrix $\rho_B$ is **unchanged**:

$$ \Large  \rho_B \;=\; \operatorname{Tr}_A\big((\Lambda_A \otimes I)[\rho_{AB}]\big) \;=\; \operatorname{Tr}_A(\rho_{AB}). $$

## So entanglement alone cannot transmit a single bit of information from Alice to Bob — quantum mechanics respects relativistic causality. (This is the formal version of "the no-cloning theorem follows from no-signalling": *if* cloning were possible, Alice could signal Bob by measurement-basis choice.)

## No-signalling is also a property of any non-signalling theory, including hypothetical *post-quantum* theories (PR-boxes). It is strictly weaker than quantum mechanics — there exist non-signalling correlations that are not reachable by any quantum strategy.

In [1]:
# Empirical no-signalling check: Bob's marginal on a Bell pair is fixed regardless of Alice's local unitary.
import numpy as np

ket0 = np.array([1, 0], dtype=complex)
ket1 = np.array([0, 1], dtype=complex)
phi_plus = (np.kron(ket0, ket0) + np.kron(ket1, ket1)) / np.sqrt(2)
rho_AB = np.outer(phi_plus, phi_plus.conj())

def partial_trace_A(rho_AB, dA=2, dB=2):
    rho_T = rho_AB.reshape(dA, dB, dA, dB)
    return np.einsum('ijik->jk', rho_T)

I = np.eye(2, dtype=complex)
H = (1/np.sqrt(2))*np.array([[1, 1], [1, -1]], dtype=complex)
X = np.array([[0, 1], [1, 0]], dtype=complex)
T = np.array([[1, 0], [0, np.exp(1j*np.pi/4)]], dtype=complex)

for name, U in [('I', I), ('H', H), ('X', X), ('T', T)]:
    U_full = np.kron(U, I)
    new_rho = U_full @ rho_AB @ U_full.conj().T
    rho_B = partial_trace_A(new_rho)
    print(f'After (U_A o I) with U = {name:1s} : rho_B =\n{np.round(rho_B, 6)}\n')

After (U_A o I) with U = I : rho_B =
[[0.5+0.j 0. +0.j]
 [0. +0.j 0.5+0.j]]

After (U_A o I) with U = H : rho_B =
[[0.5+0.j 0. +0.j]
 [0. +0.j 0.5+0.j]]

After (U_A o I) with U = X : rho_B =
[[0.5+0.j 0. +0.j]
 [0. +0.j 0.5+0.j]]

After (U_A o I) with U = T : rho_B =
[[0.5+0.j 0. +0.j]
 [0. +0.j 0.5+0.j]]



# 4. The Holevo bound

## How much *classical* information can be extracted from $n$ qubits? The intuitive guess "$2^n$ amplitudes' worth" is wildly wrong.

## **Holevo's theorem (1973).** Let Alice encode classical messages $i$ (with probabilities $p_i$) into quantum states $\rho_i$. Bob's optimal mutual information with $i$, over *any* measurement strategy, is bounded by

$$ \Large  \chi(\{p_i, \rho_i\}) \;=\; S\!\Big(\sum_i p_i \rho_i\Big) - \sum_i p_i\, S(\rho_i) \;\leq\; \log_2 d, $$

## where $S$ is von Neumann entropy and $d$ is the Hilbert-space dimension. For $n$ qubits, $\log_2 d = n$.

## **Takeaway:** $n$ qubits can carry at most $n$ classical bits of information. The exponentially many amplitudes are *internal* — measurement projects them down to $n$ bits' worth of outcome. Quantum speedups exploit the internal richness of the state *during the computation*, not the readout.

## Superdense coding does not violate Holevo: it sends $2$ classical bits per qubit *only because of a pre-shared ebit*, which itself was set up via a $2$-qubit channel. The total quantum capacity remains $1$ classical bit per qubit per use.

# 5. BBBV: Grover is optimal

## **Theorem (Bennett–Bernstein–Brassard–Vazirani, 1997).** Any quantum algorithm computing $\mathrm{OR}_N$ (equivalently: unstructured search) with bounded error makes at least

$$ \Large  \Omega(\sqrt N) $$

## queries to the oracle. So Grover's $O(\sqrt N)$ is **optimal up to a constant**.

## **Why this matters for complexity:** the same theorem implies $\mathrm{NP} \not\subseteq \mathrm{BQP}$ relative to a random oracle. Unstructured NP-hard problems (the worst-case 3-SAT, etc.) admit at most a quadratic quantum speedup. For AES-256, $\sqrt{2^{256}} = 2^{128}$ — still classically infeasible, so symmetric crypto is largely safe post-quantum.

## **Proof sketch (polynomial method).** After $T$ queries the acceptance probability is a polynomial of degree $\leq 2T$ in $x_1, \dots, x_N$. For OR, the symmetrised polynomial $p(|x|)$ (depending only on Hamming weight) must satisfy $p(0) \leq 1/3$ and $p(k) \geq 2/3$ for all $k \geq 1$. Markov's inequality forces $\deg p = \Omega(\sqrt N)$. ∎

# 6. No fast-forwarding

## **Theorem (Berry–Ahokas–Cleve–Sanders, 2007; Atia–Aharonov, 2017).** Generic Hamiltonian simulation cannot be done in time sub-linear in the simulated time $t$. Specifically: simulating $e^{-iHt}$ for a generic $H$ requires $\Omega(t)$ queries to $H$.

## So a quantum computer cannot, in general, *fast-forward* a quantum system — running 1000 seconds of dynamics still costs (at least) 1000 seconds of quantum gates.

## Special structured Hamiltonians can be fast-forwarded (commuting Hamiltonians, certain free fermions, the QFT), but the *generic* case is rigorously bounded below.

## Consequence: HHL-style algorithms gain their $\log N$ scaling not from skipping time evolution, but from reading out a clever *global* property (an expectation value) — exactly what the previous notebook called out as the genuine resource of quantum linear-algebra.

# 7. Bell inequalities as a lower bound

## **CHSH inequality.** For any *local hidden-variable* model with binary outcomes,

$$ \Large  |E(a, b) + E(a, b') + E(a', b) - E(a', b')| \;\leq\; 2. $$

## Quantum mechanics achieves $2\sqrt 2$ (Tsirelson's bound) with an entangled Bell pair and judicious measurement angles. Experimental confirmations (Aspect 1982; loophole-free 2015 — Hensen et al., Shalm et al., Giustina et al.) rule out *all* local hidden-variable theories.

## CHSH is a *lower bound* in two ways:

## - It is a lower bound on the *strangeness* required of any underlying theory: locality + realism + free will → CHSH $\leq 2$, and nature violates this.
## - It is a *communication-complexity-style* lower bound: any classical simulation of a Bell pair must use **non-local resources** — communication, post-selection, or hidden non-locality.

## **PR-boxes** (Popescu–Rohrlich, 1994) achieve CHSH = 4, the algebraic maximum. They are non-signalling but *not* quantum. Their non-existence in nature is a hint that quantum mechanics sits at a special place in the landscape of physical theories.

In [2]:
# CHSH demonstration: quantum strategy reaches 2*sqrt(2), classical caps at 2.
import numpy as np

ket0 = np.array([1, 0], dtype=complex)
ket1 = np.array([0, 1], dtype=complex)
phi_plus = (np.kron(ket0, ket0) + np.kron(ket1, ket1)) / np.sqrt(2)

def measurement_operator(theta):
    # cos(theta) Z + sin(theta) X
    Z = np.array([[1, 0], [0, -1]], dtype=complex)
    X = np.array([[0, 1], [1, 0]], dtype=complex)
    return np.cos(theta) * Z + np.sin(theta) * X

def correlation(psi, theta_A, theta_B):
    M = np.kron(measurement_operator(theta_A), measurement_operator(theta_B))
    return (psi.conj() @ M @ psi).real

# Optimal CHSH angles
a, a_prime = 0,            np.pi / 2
b, b_prime = np.pi / 4,    -np.pi / 4
S = (correlation(phi_plus, a, b)
     + correlation(phi_plus, a, b_prime)
     + correlation(phi_plus, a_prime, b)
     - correlation(phi_plus, a_prime, b_prime))
print(f'CHSH (quantum) = {S:.4f}')
print(f'2*sqrt(2)      = {2*np.sqrt(2):.4f}')
print(f'classical max  = 2.0000')

CHSH (quantum) = 2.8284
2*sqrt(2)      = 2.8284
classical max  = 2.0000


# 8. Other useful no-go results

## A short, not-exhaustive list:

## - **Quantum state discrimination.** Non-orthogonal states $|\psi\rangle, |\phi\rangle$ with $\langle\psi|\phi\rangle \neq 0$ cannot be   distinguished with certainty. Best success: $\frac{1}{2}(1 + \sqrt{1 - |\langle\psi|\phi\rangle|^2})$ (Helstrom bound).
## - **No universal NOT.** No unitary maps every $|\psi\rangle$ to its orthogonal complement $|\psi^\perp\rangle$. Optimal   approximate universal NOT has fidelity $2/3$.
## - **No deterministic entanglement swapping for unknown states.** Generalises no-cloning.
## - **Eastin–Knill theorem (2009).** No quantum error-correcting code can implement a *universal* set of gates transversally.   This is *why* magic-state distillation exists.
## - **PBR theorem (Pusey–Barrett–Rudolph, 2012).** The quantum state is "real" in any model satisfying mild independence   assumptions — *ψ*-epistemic interpretations are ruled out.
## - **Gottesman–Knill theorem.** Clifford circuits on stabilizer inputs are classically efficiently simulable —   so $T$ gates are genuinely the source of quantum advantage.

# 9. The big picture: what quantum computers cannot do

## Pulling the threads together, every realistic quantum advantage must:

## 1. **Find structure to exploit.** Unstructured search caps at $\sqrt N$; promise problems and algebraic structure unlock more.
## 2. **Avoid demanding bit-by-bit readout.** The Holevo bound says only $n$ classical bits come out of $n$ qubits per use.
## 3. **Pay for time evolution.** No fast-forwarding for generic Hamiltonians.
## 4. **Respect locality and causality.** No signalling, no cloning, no broadcasting.
## 5. **Carry a $T$-count cost in fault tolerance.** Eastin–Knill says transversal universal gates do not exist; magic state distillation is the bill.

## These are *not* engineering obstacles — they are mathematical/physical theorems. Any practical quantum algorithm we trust will live inside the box they define. The art of quantum algorithm design is finding problems whose structure happens to fit through this narrow corridor: factoring, Hamiltonian simulation, structured linear algebra, and a slowly growing list of others.

# Recap

## - **No-cloning, no-broadcasting, no-deletion.** Quantum information is not freely copyable.
## - **No-signalling.** Entanglement does not transmit information faster than light.
## - **Holevo bound.** $n$ qubits carry at most $n$ classical bits per use.
## - **BBBV.** Grover's $\sqrt N$ is optimal; NP is not in BQP relative to a random oracle.
## - **No fast-forwarding.** Generic Hamiltonian simulation costs $\Omega(t)$.
## - **CHSH / Tsirelson.** Quantum violates local-realism by exactly $\sqrt 2$.
## - **Eastin–Knill.** No universal transversal gate set ⇒ magic state distillation is unavoidable.

## **End of chapter 10.** And end of the course! The path from "complex numbers and Born's rule" to "every known quantum algorithm sits inside this corridor" is complete. From here, the natural directions are research papers — *Quantum Computation and Quantum Information* (Nielsen & Chuang), *Quantum Computing Since Democritus* (Aaronson), and the arXiv listings under **quant-ph** and **cs.CC**.